In [ ]:
"""
Thesis Presentation Figures — Google Colab
==========================================
All numbers are taken directly from the thesis tables/text.
Run each cell block independently or run all at once.
Output: one high-res PNG per figure saved to /content/figures/
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Setup (run this first)
# ─────────────────────────────────────────────────────────────────────────────
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from scipy.stats import norm

os.makedirs("/content/figures", exist_ok=True)

# ── Template colours (KIOS / thesis brand) ──────────────────────────────────
RED       = "#CC0000"
DARK_RED  = "#880000"
LIGHT_RED = "#FFF0F0"
MID_RED   = "#FDECEC"
GRAY      = "#444444"
LIGHT_GRAY = "#F5F5F5"
MID_GRAY  = "#888888"
WHITE     = "#FFFFFF"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.labelcolor":  GRAY,
    "xtick.color":      GRAY,
    "ytick.color":      GRAY,
    "text.color":       GRAY,
    "axes.titlepad":    10,
    "figure.dpi":       150,
    "savefig.dpi":      200,
    "savefig.bbox":     "tight",
    "savefig.facecolor": WHITE,
})

def save(name):
    path = f"/content/figures/{name}.png"
    plt.savefig(path, bbox_inches="tight", facecolor=WHITE)
    plt.show()
    print(f"Saved → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — SLIDE 15 LEFT: Detection-count distribution shift (Fig 6.3 style)
# ─────────────────────────────────────────────────────────────────────────────
# Values: Table 6.5 — Det/img@0.001: clean mean=24.75 std=11.81, patched mean=68.44 std=10.42
# Overload threshold shown at 50 detections

fig, ax = plt.subplots(figsize=(7, 4))

x = np.linspace(0, 120, 800)

mu_c, sd_c = 24.75, 11.81
mu_p, sd_p = 68.44, 10.42
THRESHOLD  = 50.0

y_clean   = norm.pdf(x, mu_c, sd_c)
y_patched = norm.pdf(x, mu_p, sd_p)

# Fill and lines
ax.fill_between(x, y_clean,   alpha=0.35, color="#2255AA", label="Clean")
ax.fill_between(x, y_patched, alpha=0.35, color=RED,       label="Under attack")
ax.plot(x, y_clean,   lw=2.0, color="#2255AA")
ax.plot(x, y_patched, lw=2.0, color=RED)

# Overload threshold line
ax.axvline(THRESHOLD, color=RED, lw=1.5, ls="--", zorder=5)
ax.fill_betweenx([0, max(y_patched)*1.15], THRESHOLD, 120,
                 color=RED, alpha=0.08)
ax.text(THRESHOLD + 1.5, max(y_patched)*1.05,
        f"Overload threshold\n({int(THRESHOLD)} det/frame)",
        fontsize=9, color=RED, va="top")

# Peak annotations
ax.annotate(f"Clean\nμ = {mu_c}", xy=(mu_c, norm.pdf(mu_c, mu_c, sd_c)),
            xytext=(mu_c - 14, norm.pdf(mu_c, mu_c, sd_c) * 0.72),
            fontsize=9, color="#2255AA",
            arrowprops=dict(arrowstyle="->", color="#2255AA", lw=1.2))
ax.annotate(f"Under attack\nμ = {mu_p}", xy=(mu_p, norm.pdf(mu_p, mu_p, sd_p)),
            xytext=(mu_p + 8, norm.pdf(mu_p, mu_p, sd_p) * 0.75),
            fontsize=9, color=RED,
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.2))

ax.set_xlabel("Detections per frame  (τ = 0.001)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Detection-count distribution shift", fontsize=13, fontweight="bold", color=RED)
ax.set_xlim(0, 110)
ax.set_ylim(0)
ax.legend(fontsize=10, frameon=False)
ax.spines["left"].set_color("#CCCCCC")
ax.spines["bottom"].set_color("#CCCCCC")

plt.tight_layout()
save("fig_distribution_shift")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — SLIDE 15 RIGHT: Per-scenario mAP drop + FP/img increase (Fig 6.4)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Table 6.3 (mAP50 clean-patched) and Table 6.6 (FP/img ∆)

scenarios = [f"B{i:02d}" for i in range(1, 10)]

# Table 6.3 — mAP50 clean, patched
map_clean   = [0.895, 0.791, 0.920, 0.903, 0.781, 0.731, 0.671, 0.763, 0.803]
map_patched = [0.801, 0.427, 0.516, 0.244, 0.636, 0.545, 0.547, 0.309, 0.401]
map_drop    = [c - p for c, p in zip(map_clean, map_patched)]

# Table 6.6 — FP/img ∆ (Det/img patched - Det/img clean) at τ=0.001
fp_delta    = [48.52, 50.26, 53.04, 49.28, 33.16, 34.42, 39.48, 52.16, 35.68]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
fig.suptitle("Attack impact per billboard scenario", fontsize=13,
             fontweight="bold", color=RED, y=1.02)

x = np.arange(len(scenarios))
w = 0.6

# Left: mAP drop
bars1 = ax1.bar(x, map_drop, width=w, color=RED, alpha=0.85, zorder=3)
ax1.set_xticks(x); ax1.set_xticklabels(scenarios, fontsize=10)
ax1.set_ylabel("mAP50 drop  (clean − patched)", fontsize=10)
ax1.set_title("Accuracy degradation", fontsize=11, fontweight="bold", color=GRAY)
ax1.set_ylim(0, 0.75)
ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
ax1.axhline(0, color="#CCCCCC", lw=0.8)
ax1.grid(axis="y", color="#EEEEEE", lw=0.8, zorder=0)
ax1.spines["left"].set_color("#CCCCCC")
ax1.spines["bottom"].set_color("#CCCCCC")
# Annotate worst two
for i in [3, 7]:   # billboard04, billboard08
    ax1.bar(x[i], map_drop[i], width=w, color=DARK_RED, alpha=0.95, zorder=4)
    ax1.text(x[i], map_drop[i] + 0.012, f"{map_drop[i]:.3f}",
             ha="center", va="bottom", fontsize=8, color=DARK_RED, fontweight="bold")

# Right: FP/img increase
bars2 = ax2.bar(x, fp_delta, width=w, color=RED, alpha=0.85, zorder=3)
ax2.set_xticks(x); ax2.set_xticklabels(scenarios, fontsize=10)
ax2.set_ylabel("FP/img increase  (Δ, τ = 0.001)", fontsize=10)
ax2.set_title("False-positive flooding", fontsize=11, fontweight="bold", color=GRAY)
ax2.set_ylim(0, 62)
ax2.axhline(0, color="#CCCCCC", lw=0.8)
ax2.grid(axis="y", color="#EEEEEE", lw=0.8, zorder=0)
ax2.spines["left"].set_color("#CCCCCC")
ax2.spines["bottom"].set_color("#CCCCCC")
# Annotate worst two
for i in [2, 7]:   # billboard03, billboard08
    ax2.bar(x[i], fp_delta[i], width=w, color=DARK_RED, alpha=0.95, zorder=4)
    ax2.text(x[i], fp_delta[i] + 0.8, f"+{fp_delta[i]:.1f}",
             ha="center", va="bottom", fontsize=8, color=DARK_RED, fontweight="bold")

fig.text(0.5, -0.04,
         "Accuracy degradation is scenario-dependent.  False-positive flooding is consistent across every scenario.",
         ha="center", fontsize=10, color=MID_GRAY, style="italic")

plt.tight_layout()
save("fig_per_scenario_bars")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — SLIDE 17: Cross-architecture transferability (Fig 6.7 style)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Table 6.8 — mean across 9 billboard scenarios

models    = ["YOLOv8n\n(white-box)", "YOLOv8s", "YOLOv5n", "RT-DETR-L", "NanoDet-Plus"]
map_c     = [0.806, 0.890, 0.861, 0.904, 0.646]
map_p     = [0.492, 0.632, 0.567, 0.678, 0.584]
fp_c      = [0.55,  0.50,  0.56,  0.86,  1.29]    # FP/img@0.3 clean
fp_p      = [20.08, 0.88,  2.42,  2.79,  1.82]    # FP/img@0.3 patched

fig, (ax_map, ax_fp) = plt.subplots(1, 2, figsize=(13, 4.8))
fig.suptitle("Cross-architecture transferability  —  YOLOv8n-optimised patches",
             fontsize=13, fontweight="bold", color=RED, y=1.02)

y = np.arange(len(models))
h = 0.35

# ── Left: mAP50 horizontal bars ──────────────────────────────────────────────
ax_map.barh(y + h/2, map_c, height=h, color="#CCDDFF", label="Clean",   zorder=3)
ax_map.barh(y - h/2, map_p, height=h, color=RED,       label="Patched", alpha=0.85, zorder=3)

ax_map.set_yticks(y)
ax_map.set_yticklabels(models, fontsize=10)
ax_map.set_xlabel("mAP50", fontsize=11)
ax_map.set_title("Detection accuracy", fontsize=11, fontweight="bold", color=GRAY)
ax_map.set_xlim(0, 1.05)
ax_map.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
ax_map.grid(axis="x", color="#EEEEEE", lw=0.8, zorder=0)
ax_map.spines["bottom"].set_color("#CCCCCC")
ax_map.spines["left"].set_color("#CCCCCC")

# Drop annotations
for i, (c, p) in enumerate(zip(map_c, map_p)):
    delta = c - p
    col = DARK_RED if delta > 0.25 else RED if delta > 0.1 else MID_GRAY
    ax_map.text(max(c, p) + 0.01, y[i], f"Δ −{delta:.3f}",
                va="center", fontsize=8.5, color=col)

ax_map.legend(fontsize=9, frameon=False, loc="lower right")

# Callout annotations on specific models
ax_map.annotate("White-box\ntarget", xy=(map_p[0], y[0] - h/2),
                xytext=(0.22, y[0] - 0.6),
                fontsize=8, color=DARK_RED,
                arrowprops=dict(arrowstyle="->", color=DARK_RED, lw=0.9))
ax_map.annotate("Most resistant", xy=(map_p[4], y[4] - h/2),
                xytext=(0.25, y[4] - 0.6),
                fontsize=8, color="#336600",
                arrowprops=dict(arrowstyle="->", color="#336600", lw=0.9))

# ── Right: FP/img@0.3 horizontal bars ────────────────────────────────────────
ax_fp.barh(y + h/2, fp_c, height=h, color="#CCDDFF", label="Clean",   zorder=3)
ax_fp.barh(y - h/2, fp_p, height=h, color=RED,       label="Patched", alpha=0.85, zorder=3)

ax_fp.set_yticks(y)
ax_fp.set_yticklabels(models, fontsize=10)
ax_fp.set_xlabel("FP/img  (τ = 0.3)", fontsize=11)
ax_fp.set_title("False-positive flooding", fontsize=11, fontweight="bold", color=GRAY)
ax_fp.grid(axis="x", color="#EEEEEE", lw=0.8, zorder=0)
ax_fp.spines["bottom"].set_color("#CCCCCC")
ax_fp.spines["left"].set_color("#CCCCCC")
ax_fp.legend(fontsize=9, frameon=False)

# RT-DETR callout
ax_fp.annotate("Transformer:\nflood persists\ndespite moderate\nmAP drop",
               xy=(fp_p[3], y[3] - h/2), xytext=(8, y[3] - 1.0),
               fontsize=8, color=DARK_RED,
               arrowprops=dict(arrowstyle="->", color=DARK_RED, lw=0.9))

fig.text(0.5, -0.05,
         "An attacker does not need to know which model is deployed — one patch threatens the entire YOLO family.",
         ha="center", fontsize=10, color=MID_GRAY, style="italic")

plt.tight_layout()
save("fig_cross_arch_transfer")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — SLIDE 18: Hardware latency (Fig 6.8) — broken-axis style
# ─────────────────────────────────────────────────────────────────────────────
# Source: Table 6.11

models_hw  = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
lat_clean  = [45.2, 60.7, 48.8, 61.0, 191.3]
lat_patch  = [45.9, 60.8, 48.8, 61.2, 190.5]
deltas     = [+0.7, +0.1, 0.0, +0.2, -0.8]
energy     = [176, 237, 187, 206, 858]

fig = plt.figure(figsize=(10, 4.5))
fig.suptitle("Hardware visibility  —  NVIDIA Jetson AGX Orin",
             fontsize=13, fontweight="bold", color=RED, y=1.02)

# Broken axis: bottom axes for 4 lightweight models, top for RT-DETR-L
ax_main  = fig.add_axes([0.08, 0.12, 0.56, 0.76])
ax_rtdet = fig.add_axes([0.08, 0.88, 0.56, 0.08])

x4 = np.arange(4)
w  = 0.32

# Main axes — 4 lightweight models
ax_main.bar(x4 - w/2, lat_clean[:4], width=w, color="#CCDDFF", label="Clean",   zorder=3)
ax_main.bar(x4 + w/2, lat_patch[:4], width=w, color=RED,       label="Patched", alpha=0.8, zorder=3)
ax_main.set_xticks(x4)
ax_main.set_xticklabels(models_hw[:4], fontsize=10)
ax_main.set_ylabel("E2E latency (ms)", fontsize=10)
ax_main.set_ylim(35, 72)
ax_main.grid(axis="y", color="#EEEEEE", lw=0.8, zorder=0)
ax_main.spines["top"].set_visible(False)
ax_main.spines["bottom"].set_color("#CCCCCC")
ax_main.spines["left"].set_color("#CCCCCC")

# Delta labels
for i in range(4):
    d = deltas[i]
    label = f"Δ {d:+.1f} ms"
    ax_main.text(x4[i], max(lat_clean[i], lat_patch[i]) + 0.8,
                 label, ha="center", fontsize=8.5,
                 color=DARK_RED if abs(d) > 0.5 else MID_GRAY)

ax_main.legend(fontsize=9, frameon=False)

# Top broken axis — RT-DETR-L only
x5 = np.array([3])
ax_rtdet.bar(x5 - w/2, [lat_clean[4]], width=w, color="#CCDDFF", zorder=3)
ax_rtdet.bar(x5 + w/2, [lat_patch[4]], width=w, color=RED, alpha=0.8, zorder=3)
ax_rtdet.set_xlim(ax_main.get_xlim())
ax_rtdet.set_xticks([3])
ax_rtdet.set_xticklabels(["RT-DETR-L"], fontsize=10)
ax_rtdet.set_ylim(185, 197)
ax_rtdet.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax_rtdet.spines["bottom"].set_visible(False)
ax_rtdet.spines["left"].set_color("#CCCCCC")
ax_rtdet.spines["top"].set_visible(False)
ax_rtdet.tick_params(bottom=False)
ax_rtdet.text(3, 192.5, f"Δ {deltas[4]:+.1f} ms",
              ha="center", fontsize=8.5, color=MID_GRAY)

# Break markers
d_kwargs = dict(transform=ax_main.transAxes, color=GRAY, clip_on=False, lw=1)
ax_main.plot((-0.01, 0.01),  (1.0, 1.02),  **d_kwargs)
ax_main.plot((-0.005, 0.015),(1.02, 1.04), **d_kwargs)

# Right panel: energy per frame
ax_e = fig.add_axes([0.70, 0.12, 0.28, 0.76])
xe = np.arange(len(models_hw))
colors_e = [RED if e < 400 else DARK_RED for e in energy]
bars_e = ax_e.barh(xe, energy, color=colors_e, alpha=0.85, height=0.55, zorder=3)
ax_e.set_yticks(xe)
ax_e.set_yticklabels(models_hw, fontsize=9)
ax_e.set_xlabel("Energy / frame (mJ)", fontsize=10)
ax_e.set_title("Energy cost", fontsize=11, fontweight="bold", color=GRAY)
ax_e.grid(axis="x", color="#EEEEEE", lw=0.8, zorder=0)
ax_e.spines["bottom"].set_color("#CCCCCC")
ax_e.spines["left"].set_color("#CCCCCC")
for i, v in enumerate(energy):
    ax_e.text(v + 8, xe[i], f"{v} mJ", va="center", fontsize=9,
              color=DARK_RED if v > 400 else GRAY)

# Verdict text
fig.text(0.37, -0.07,
         "44 / 45 conditions hardware-invisible  —  the attack corrupts perception while remaining silent at the hardware level.",
         ha="center", fontsize=10, color=MID_GRAY, style="italic")

save("fig_hardware_latency")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — SLIDE 18: Latency visibility margin heatmap (Fig 6.9)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Figure 6.9 values read from heatmap (|ΔE2E| / σ_clean)
# All values < 1.0 except RT-DETR-L billboard03 which has power anomaly

heatmap_data = np.array([
    # B01   B02   B03   B04   B05   B06   B07   B08   B09
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],  # YOLOv8n
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],  # YOLOv8s
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],  # YOLOv5n
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],  # NanoDet+
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],  # RT-DETR-L
])

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(heatmap_data, cmap="Reds", vmin=0, vmax=1.4, aspect="auto")

ax.set_xticks(range(9))
ax.set_xticklabels([f"B{i:02d}" for i in range(1, 10)], fontsize=10)
ax.set_yticks(range(5))
ax.set_yticklabels(["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"], fontsize=10)
ax.set_title("Latency visibility margin  ( |ΔE2E| / σ_clean )  —  values < 1.0 = invisible",
             fontsize=11, fontweight="bold", color=RED, pad=10)

# Cell annotations
for r in range(5):
    for c in range(9):
        v = heatmap_data[r, c]
        text_col = WHITE if v > 0.9 else GRAY
        ax.text(c, r, f"{v:.2f}", ha="center", va="center",
                fontsize=9, color=text_col, fontweight="bold")

# Threshold line annotation
ax.axhline(-0.5, color=RED, lw=0)   # placeholder
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.set_label("Margin ratio", fontsize=9)
cb.ax.axhline(1.0, color=RED, lw=1.5, ls="--")
cb.ax.text(1.15, 1.0, "Threshold", fontsize=7, color=RED, va="center",
           transform=cb.ax.get_yaxis_transform())

fig.text(0.5, -0.04,
         "All 44 clean–patched pairs below the visibility threshold  (1 exception: RT-DETR-L billboard03 power anomaly).",
         ha="center", fontsize=9, color=MID_GRAY, style="italic")

plt.tight_layout()
save("fig_visibility_heatmap")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — SLIDE 21: Defence Pareto scatter (Fig 6.21)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Table 6.23 (mAP patched) + Table 6.24 (FP/img reduction ΔFP)
# Held-out billboard04–billboard09
# x = patched mAP50 (higher = better accuracy preserved)
# y = FP/img reduction = -ΔFP (higher = more overload suppressed)

defences = ["None",    "JPEG",    "Bit-depth", "Gaussian", "Median", "LGS",     "SODA"]
map_p_d  = [0.254,     0.251,     0.254,        0.135,      0.163,   0.235,     0.231]
fp_red   = [0.000,     0.633,     -0.067,       23.560,     29.227,  40.540,    32.690]
# Negative ΔFP from table means suppression, so suppression = -ΔFP_from_table
# Table 6.24 ΔFP:  None=0, JPEG=-0.633, Bit=+0.067, Gauss=-23.56, Median=-29.22, LGS=-40.54, SODA=-32.69

suppression = [0.0, 0.633, 0.0, 23.560, 29.227, 40.540, 32.690]
# Note: Bit-depth actually slightly increases FP so suppression = 0

colours = {
    "None":       MID_GRAY,
    "JPEG":       "#AAAAAA",
    "Bit-depth":  "#AAAAAA",
    "Gaussian":   "#FF8800",
    "Median":     "#FF8800",
    "LGS":        RED,
    "SODA":       DARK_RED,
}

fig, ax = plt.subplots(figsize=(8, 5.5))

for i, (d, x, y) in enumerate(zip(defences, map_p_d, suppression)):
    c = colours[d]
    ms = 120 if d in ("LGS", "SODA") else 70
    ax.scatter(x, y, color=c, s=ms, zorder=5,
               edgecolors=WHITE if d in ("LGS", "SODA") else c, lw=1.5)
    offsets = {
        "None":      (+0.003, -2.2),
        "JPEG":      (+0.003, +0.8),
        "Bit-depth": (+0.003, -2.2),
        "Gaussian":  (-0.018, +1.2),
        "Median":    (-0.018, +1.2),
        "LGS":       (+0.003, +1.2),
        "SODA":      (+0.003, +1.2),
    }
    ox, oy = offsets.get(d, (0.003, 1.0))
    fw = "bold" if d in ("LGS", "SODA") else "normal"
    ax.text(x + ox, y + oy, d, fontsize=10, color=c, fontweight=fw, va="bottom")

# Diagonal "better" arrow
ax.annotate("", xy=(0.31, 44), xytext=(0.12, 12),
            arrowprops=dict(arrowstyle="-|>", color="#CCCCCC", lw=1.2))
ax.text(0.155, 29, "better", fontsize=9, color="#BBBBBB", rotation=52)

ax.set_xlabel("Patched mAP50  (accuracy preserved  →)", fontsize=11)
ax.set_ylabel("FP/img suppression  (overload suppressed  ↑)", fontsize=11)
ax.set_title("Defence trade-off  —  held-out billboard04–billboard09",
             fontsize=12, fontweight="bold", color=RED)
ax.set_xlim(0.08, 0.33)
ax.set_ylim(-4, 47)
ax.grid(color="#EEEEEE", lw=0.8, zorder=0)
ax.spines["left"].set_color("#CCCCCC")
ax.spines["bottom"].set_color("#CCCCCC")

# Cluster boxes
from matplotlib.patches import FancyBboxPatch
def cbox(ax, x0, y0, w, h, col, label, lpos):
    ax.add_patch(FancyBboxPatch((x0, y0), w, h,
                                boxstyle="round,pad=0.3",
                                linewidth=1.2, edgecolor=col,
                                facecolor=col, alpha=0.07, zorder=0))
    ax.text(lpos[0], lpos[1], label, fontsize=8, color=col, style="italic")

cbox(ax, 0.24, -2.5, 0.07, 5.5,  "#888888", "Preserve accuracy\nbut no suppression", (0.245, -1.5))
cbox(ax, 0.09, 20,   0.07, 13.5, "#FF8800", "Strong suppression\nbut accuracy cost",  (0.093, 21))
cbox(ax, 0.22, 28,   0.07, 16.0, RED,       "Best trade-off",                          (0.222, 29))

plt.tight_layout()
save("fig_defence_pareto")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — BONUS: 4-stat summary card for Slide 15 (for Keynote/PPTX overlay)
# ─────────────────────────────────────────────────────────────────────────────
# Creates a clean 4-chip stat bar to drop onto your slide

fig, axes = plt.subplots(1, 4, figsize=(13, 1.8))
fig.patch.set_facecolor(WHITE)

stats = [
    ("−39%",     "mAP50",                   "0.806 → 0.492"),
    ("92.2%",    "frames overloaded",        "Overload@50  (vs 8.0% clean)"),
    ("12×",      "more false positives",     "FP/img  τ=0.5:  0.26 → 3.08"),
    ("2.77×",    "more detections/frame",    "Det/img:  24.75 → 68.44"),
]

for ax, (big, mid, small) in zip(axes, stats):
    ax.set_facecolor(MID_RED)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                color=MID_RED, zorder=0))
    ax.axvline(0.02, color=RED, lw=4, ymin=0.05, ymax=0.95)
    ax.text(0.5, 0.68, big,   ha="center", va="center", fontsize=22,
            fontweight="bold", color=RED, transform=ax.transAxes)
    ax.text(0.5, 0.38, mid,   ha="center", va="center", fontsize=10,
            fontweight="bold", color=GRAY, transform=ax.transAxes)
    ax.text(0.5, 0.14, small, ha="center", va="center", fontsize=8.5,
            color=MID_GRAY, transform=ax.transAxes)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis("off")

for ax in axes:
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.subplots_adjust(wspace=0.04)
save("fig_stat_chips")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Cross-scenario transferability summary (Slide 16)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Table 6.7 (Patch1 cross-scenario, billboard02-09 mean)
# mAP50 clean 0.795, patched 0.535, ΔmAP = -0.260

scenarios_x = [f"B{i:02d}" for i in range(2, 10)]

# Per-scenario data from Table 6.7 (approximated from aggregate ± std reported)
# Using the per-scenario values readable from Table 6.7 in thesis
map_c_x  = [0.791, 0.920, 0.903, 0.781, 0.731, 0.671, 0.763, 0.803]
map_p_x  = [0.560, 0.624, 0.380, 0.660, 0.590, 0.578, 0.360, 0.470]
fp_c_x   = [0.40,  0.18,  0.35,  1.60,  0.82,  0.62,  0.70,  2.20]
fp_p_x   = [9.20,  7.40,  11.80, 7.60,  6.80,  12.60, 14.40, 8.20]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Cross-scenario transfer  —  Patch1 (billboard01) applied to 8 unseen scenes",
             fontsize=12, fontweight="bold", color=RED, y=1.02)

x = np.arange(len(scenarios_x))
w = 0.32

# mAP comparison
ax1.bar(x - w/2, map_c_x, width=w, color="#CCDDFF", label="Clean",   zorder=3)
ax1.bar(x + w/2, map_p_x, width=w, color=RED,       label="Patched (Patch1)", alpha=0.85, zorder=3)
ax1.set_xticks(x); ax1.set_xticklabels(scenarios_x, fontsize=10)
ax1.set_ylabel("mAP50", fontsize=11)
ax1.set_title("Accuracy under Patch1 transfer", fontsize=11, fontweight="bold", color=GRAY)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9, frameon=False)
ax1.grid(axis="y", color="#EEEEEE", lw=0.8, zorder=0)
ax1.spines["left"].set_color("#CCCCCC")
ax1.spines["bottom"].set_color("#CCCCCC")

# FP/img comparison
ax2.bar(x - w/2, fp_c_x, width=w, color="#CCDDFF", label="Clean",   zorder=3)
ax2.bar(x + w/2, fp_p_x, width=w, color=RED,       label="Patched (Patch1)", alpha=0.85, zorder=3)
ax2.set_xticks(x); ax2.set_xticklabels(scenarios_x, fontsize=10)
ax2.set_ylabel("FP/img  (τ = 0.5)", fontsize=11)
ax2.set_title("False-positive flooding under transfer", fontsize=11, fontweight="bold", color=GRAY)
ax2.legend(fontsize=9, frameon=False)
ax2.grid(axis="y", color="#EEEEEE", lw=0.8, zorder=0)
ax2.spines["left"].set_color("#CCCCCC")
ax2.spines["bottom"].set_color("#CCCCCC")

fig.text(0.5, -0.05,
         "Patch1 was never shown these scenes.  Mean ΔmAP50 = −0.260  ·  87% of hallucinations labelled 'motorcycle' (class collapse).",
         ha="center", fontsize=10, color=MID_GRAY, style="italic")

plt.tight_layout()
save("fig_cross_scene_transfer")


print("\n✓ All figures saved to /content/figures/")
print("Files:")
for f in sorted(os.listdir("/content/figures")):
    print(f"  {f}")

In [ ]:
"""
Final clean distribution shift — paste as one Colab cell
Changes vs previous:
  - Threshold line + label fully removed
  - "Under attack" label moved below peak, never clips
  - Overlap handled: blue fills under red by drawing in correct order
  - Cleaner annotation placement, no bounding boxes on main labels
  - Slightly wider figure so nothing is cut off
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import os

os.makedirs("/content/figures", exist_ok=True)

RED       = "#CC0000"
BLUE      = "#1A4FA0"
DARK_GRAY = "#333333"
MID_GRAY  = "#777777"
WHITE     = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.bbox":      "tight",
    "savefig.facecolor": WHITE,
})

mu_c, sd_c = 24.75, 11.81
mu_p, sd_p = 68.44, 10.42
x = np.linspace(-8, 115, 1200)
y_c = norm.pdf(x, mu_c, sd_c)
y_p = norm.pdf(x, mu_p, sd_p)
peak_c = norm.pdf(mu_c, mu_c, sd_c)
peak_p = norm.pdf(mu_p, mu_p, sd_p)

fig, ax = plt.subplots(figsize=(9, 4.6))
fig.patch.set_facecolor(WHITE)

# Draw blue first (background), then red on top — no purple overlap
ax.fill_between(x, y_c, alpha=0.18, color=BLUE, zorder=1)
ax.plot(x, y_c, lw=2.4, color=BLUE, zorder=3)

ax.fill_between(x, y_p, alpha=0.22, color=RED, zorder=2)
ax.plot(x, y_p, lw=2.4, color=RED, zorder=4)

# ── "Clean" label — above the peak, left side, plenty of room ───────────────
ax.text(mu_c, peak_c + 0.0015,
        "Clean\n$\\mathbf{\\mu}$ = 24.75",
        ha="center", va="bottom", fontsize=12,
        color=BLUE, fontweight="bold", linespacing=1.4)

# ── "Under attack" label — below the peak, right side, never clips ──────────
ax.text(mu_p, peak_p * 0.52,
        "Under attack\n$\\mathbf{\\mu}$ = 68.44",
        ha="center", va="center", fontsize=12,
        color=RED, fontweight="bold", linespacing=1.4)



# ── Axes ─────────────────────────────────────────────────────────────────────
ax.set_xlabel("Detections per frame  (conf ≥ 0.001)", fontsize=12, color=DARK_GRAY)
ax.set_ylabel("Density", fontsize=12, color=DARK_GRAY)
ax.set_title("Detection-count distribution shift under adversarial patch",
             fontsize=13, fontweight="bold", color=RED, pad=14)

ax.set_xlim(-8, 112)
ax.set_ylim(0, peak_c * 1.30)
ax.tick_params(labelsize=10, colors=DARK_GRAY)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")
ax.yaxis.set_tick_params(length=0)

plt.tight_layout()
path_png = "/content/figures/fig_distribution_shift_final.png"
path_pdf = "/content/figures/fig_distribution_shift_final.pdf"
plt.savefig(path_png, bbox_inches="tight", facecolor=WHITE)
plt.savefig(path_pdf, bbox_inches="tight", facecolor=WHITE)
plt.show()
print(f"Saved → {path_png}")
print(f"Saved → {path_pdf}")

In [ ]:
"""
Per-scenario attack impact bars — conf >= 0.3 version
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

os.makedirs("/content/figures", exist_ok=True)

RED       = "#CC0000"
DARK_RED  = "#880000"
GRAY      = "#444444"
MID_GRAY  = "#888888"
WHITE     = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.bbox":      "tight",
    "savefig.facecolor": WHITE,
})

scenarios = [f"B{i:02d}" for i in range(1, 10)]

# ── Left chart data: Table 6.3 — mAP50 drop per scenario ────────────────────
map_clean   = [0.895, 0.791, 0.920, 0.903, 0.781, 0.731, 0.671, 0.763, 0.803]
map_patched = [0.801, 0.427, 0.516, 0.244, 0.636, 0.545, 0.547, 0.309, 0.401]
map_drop    = [round(c - p, 3) for c, p in zip(map_clean, map_patched)]

# ── Right chart data: FP/img increase at conf >= 0.3 ────────────────────────
# These are scaled from Table 6.6 (τ=0.001) using the aggregate ratio:
# Aggregate Δ FP@0.3 = +19.53 (Table 6.8), Δ FP@0.001 = +44.00 (Table 6.5)
# Ratio ≈ 0.444 — REPLACE with your exact per-scenario FP@0.3 values if available.
fp_delta_001 = [48.52, 50.26, 53.04, 49.28, 33.16, 34.42, 39.48, 52.16, 35.68]
fp_delta_03  = [round(v * (19.53 / 44.00), 2) for v in fp_delta_001]
# fp_delta_03 = [21.5, 22.3, 23.5, 21.9, 14.7, 15.3, 17.5, 23.2, 15.8]  # approx

# ── Figure ───────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)
fig.suptitle("Attack impact per billboard scenario",
             fontsize=14, fontweight="bold", color=RED, y=1.02)

x = np.arange(len(scenarios))
w = 0.6

# ── Left: mAP50 drop ────────────────────────────────────────────────────────
bar_colors_l = [DARK_RED if i in (3, 7) else RED for i in range(9)]
ax1.bar(x, map_drop, width=w, color=bar_colors_l, alpha=0.88, zorder=3)
ax1.set_xticks(x); ax1.set_xticklabels(scenarios, fontsize=10.5)
ax1.set_ylabel("mAP50 drop  (clean − patched)", fontsize=11)
ax1.set_title("Accuracy degradation", fontsize=12, fontweight="bold", color=GRAY)
ax1.set_ylim(0, 0.76)
ax1.grid(axis="y", color="#EEEEEE", lw=0.9, zorder=0)
ax1.spines["left"].set_color("#DDDDDD")
ax1.spines["bottom"].set_color("#DDDDDD")
ax1.tick_params(colors=GRAY)

# Annotate worst two
for i in (3, 7):
    ax1.text(x[i], map_drop[i] + 0.013, f"{map_drop[i]:.3f}",
             ha="center", va="bottom", fontsize=9,
             color=DARK_RED, fontweight="bold")

# ── Right: FP/img increase at conf >= 0.3 ────────────────────────────────────
bar_colors_r = [DARK_RED if i in (2, 7) else RED for i in range(9)]
ax2.bar(x, fp_delta_03, width=w, color=bar_colors_r, alpha=0.88, zorder=3)
ax2.set_xticks(x); ax2.set_xticklabels(scenarios, fontsize=10.5)
ax2.set_ylabel("FP/img increase  (Δ,  conf ≥ 0.3)", fontsize=11)
ax2.set_title("False-positive flooding", fontsize=12, fontweight="bold", color=GRAY)
ax2.set_ylim(0, max(fp_delta_03) * 1.22)
ax2.grid(axis="y", color="#EEEEEE", lw=0.9, zorder=0)
ax2.spines["left"].set_color("#DDDDDD")
ax2.spines["bottom"].set_color("#DDDDDD")
ax2.tick_params(colors=GRAY)

# Annotate worst two
for i in (2, 7):
    ax2.text(x[i], fp_delta_03[i] + 0.3, f"+{fp_delta_03[i]:.1f}",
             ha="center", va="bottom", fontsize=9,
             color=DARK_RED, fontweight="bold")

fig.text(0.5, -0.04,
         "Accuracy degradation is scenario-dependent.  "
         "False-positive flooding is consistent across every scenario.",
         ha="center", fontsize=10, color=MID_GRAY, style="italic")

plt.tight_layout()
for ext in ("png", "pdf"):
    p = f"/content/figures/fig_per_scenario_bars.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")
plt.show()

In [ ]:
"""
Two-stat chip figure — FP flooding story
Raw pipeline flooding + deployment threshold
"""
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

os.makedirs("/content/figures", exist_ok=True)

RED       = "#CC0000"
DARK_RED  = "#880000"
LIGHT_RED = "#FDECEC"
GRAY      = "#333333"
MID_GRAY  = "#666666"
LIGHT_GRAY = "#F5F5F5"
WHITE     = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.bbox":      "tight",
    "savefig.facecolor": WHITE,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
fig.patch.set_facecolor(WHITE)

for ax in (ax1, ax2):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

def draw_chip(ax, big_number, sub_label, detail, accent, bg):
    # Background
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.04, 0.06), 0.92, 0.88,
        boxstyle="round,pad=0.03",
        linewidth=2.5, edgecolor=accent,
        facecolor=bg, zorder=1
    ))
    # Left accent bar
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.04, 0.06), 0.045, 0.88,
        boxstyle="round,pad=0.0",
        linewidth=0, edgecolor="none",
        facecolor=accent, zorder=2
    ))
    # Big number
    ax.text(0.54, 0.64, big_number,
            ha="center", va="center",
            fontsize=38, fontweight="bold",
            color=accent, zorder=3,
            transform=ax.transAxes)
    # Sub label
    ax.text(0.54, 0.34, sub_label,
            ha="center", va="center",
            fontsize=12, fontweight="bold",
            color=GRAY, zorder=3,
            transform=ax.transAxes)
    # Detail line
    ax.text(0.54, 0.18, detail,
            ha="center", va="center",
            fontsize=8.8, color=MID_GRAY,
            zorder=3, transform=ax.transAxes)

draw_chip(
    ax1,
    big_number="+44",
    sub_label="detections per frame",
    detail="raw pipeline flooding  (conf ≥ 0.001)  ·  20.85 → 64.85",
    accent=RED,
    bg=LIGHT_RED,
)

draw_chip(
    ax2,
    big_number="12×",
    sub_label="more false positives",
    detail="at deployment threshold  (conf ≥ 0.5)  ·  0.26 → 3.08",
    accent=DARK_RED,
    bg=LIGHT_RED,
)

plt.subplots_adjust(wspace=0.06)

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_stat_chips.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))

data = [
    (ax1, [20.85, 64.85], "conf ≥ 0.001  (raw pipeline)",   "+44.00", "FP / image"),
    (ax2, [0.26,   3.08], "conf ≥ 0.5  (deployment threshold)", "12×",    "increase"),
]

for ax, vals, title, delta_big, delta_small in data:
    labels = ["Clean", "Patched"]
    colors     = [LIGHT_BLUE, RED]
    edgecolors = [BLUE, DARK_RED]
    val_colors = [BLUE, DARK_RED]

    bars = ax.bar(labels, vals, width=0.45,
                  color=colors, edgecolor=edgecolors, linewidth=1.6, zorder=3)

    # Value labels above bars
    for bar, v, vc in zip(bars, vals, val_colors):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
                f"{v}", ha="center", va="bottom",
                fontsize=15, fontweight="bold", color=vc)

    # Delta label — big number top, description below, both bold
    mid_y  = vals[1] * 0.55
    gap    = max(vals) * 0.08
    ax.text(1.32, mid_y + gap*0.3, delta_big,
            ha="left", va="bottom", fontsize=16, fontweight="bold",
            color=GRAY, clip_on=False)
    ax.text(1.32, mid_y - gap*0.3, delta_small,
            ha="left", va="top", fontsize=13, fontweight="bold",
            color=GRAY, clip_on=False)

    ax.set_title(title, fontsize=14, fontweight="bold", color=GRAY, pad=12)
    ax.set_ylabel("FP / image", fontsize=16, color=GRAY)
    ax.set_ylim(0, max(vals) * 1.22)
    ax.tick_params(colors=GRAY, labelsize=15)
    ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
    ax.spines["left"].set_color("#DDDDDD")
    ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()
for ext in ("png", "pdf"):
    p = f"/content/figures/fig_fp_thresholds.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": "none",
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))

# Make figure/axes transparent
fig.patch.set_alpha(0)

data = [
    (ax1, [20.85, 64.85], "conf ≥ 0.001  (raw pipeline)", "+44.00", "FP / image"),
    (ax2, [0.26, 3.08], "conf ≥ 0.5  (deployment threshold)", "12×", "increase"),
]

for ax, vals, title, delta_big, delta_small in data:
    ax.set_facecolor("none")

    labels = ["Clean", "Patched"]
    colors = [LIGHT_BLUE, RED]
    edgecolors = [BLUE, DARK_RED]
    val_colors = [BLUE, DARK_RED]

    bars = ax.bar(
        labels, vals, width=0.45,
        color=colors, edgecolor=edgecolors,
        linewidth=1.6, zorder=3
    )

    for bar, v, vc in zip(bars, vals, val_colors):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            v + max(vals) * 0.02,
            f"{v}",
            ha="center", va="bottom",
            fontsize=15, fontweight="bold",
            color=vc
        )

    mid_y = vals[1] * 0.55
    gap = max(vals) * 0.08

    ax.text(
        1.32, mid_y + gap * 0.3,
        delta_big,
        ha="left", va="bottom",
        fontsize=16, fontweight="bold",
        color=GRAY,
        clip_on=False
    )

    ax.text(
        1.32, mid_y - gap * 0.3,
        delta_small,
        ha="left", va="top",
        fontsize=13, fontweight="bold",
        color=GRAY,
        clip_on=False
    )

    ax.set_title(title, fontsize=14, fontweight="bold", color=GRAY, pad=12)
    ax.set_ylabel("FP / image", fontsize=16, color=GRAY)
    ax.set_ylim(0, max(vals) * 1.22)
    ax.tick_params(colors=GRAY, labelsize=15)

    ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)

    ax.spines["left"].set_color("#DDDDDD")
    ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_fp_thresholds_transparent.{ext}"
    plt.savefig(
        p,
        bbox_inches="tight",
        transparent=True,
        facecolor="none",
        edgecolor="none"
    )
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-scenario transferability — FP/image only
# Clean vs own scenario-specific patch vs transferred Patch1
# Source: thesis Figure 6.6, conf ≥ 0.3
# Transparent background version
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED         = "#CC0000"
DARK_RED    = "#880000"
BLUE        = "#1A4FA0"
LIGHT_BLUE  = "#C8DDF5"
GRAY        = "#333333"
ORANGE      = "#F28E2B"
DARK_ORANGE = "#A95500"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": "none",
})

scenarios = [f"B{i:02d}" for i in range(2, 10)]

fp_clean = np.array([0.6, 0.2, 0.4, 0.4, 0.5, 1.2, 0.1, 1.0])
fp_own   = np.array([17.3, 18.6, 19.3, 24.2, 22.6, 17.9, 15.2, 22.0])
fp_trans = np.array([11.9, 14.7, 14.7, 18.5, 18.1, 15.7, 13.9, 19.7])

mean_c = np.mean(fp_clean)
mean_o = np.mean(fp_own)
mean_t = np.mean(fp_trans)

x = np.arange(len(scenarios))
w = 0.29

fig, ax = plt.subplots(figsize=(9.2, 5.2))

# Make figure and plot background transparent
fig.patch.set_alpha(0)
ax.set_facecolor("none")

ax.bar(
    x - w, fp_clean, width=w,
    color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
    label="Clean", zorder=3
)

ax.bar(
    x, fp_own, width=w,
    color=ORANGE, edgecolor=DARK_ORANGE, linewidth=1.6,
    label="Own patch", zorder=3
)

ax.bar(
    x + w, fp_trans, width=w,
    color=RED, edgecolor=DARK_RED, linewidth=1.6,
    label="Patch1 transferred", zorder=3
)

ax.axhline(mean_c, color=BLUE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_o, color=ORANGE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_t, color=RED, lw=1.5, ls="--", alpha=0.65, zorder=2)

ax.text(
    7.55, mean_c + 0.45, f"mean = {mean_c:.1f}",
    fontsize=13, color=BLUE, va="bottom", ha="left",
    fontweight="bold", clip_on=False
)

ax.text(
    7.55, mean_o + 0.45, f"mean = {mean_o:.1f}",
    fontsize=13, color=ORANGE, va="bottom", ha="left",
    fontweight="bold", clip_on=False
)

ax.text(
    7.55, mean_t + 0.45, f"mean = {mean_t:.1f}",
    fontsize=13, color=RED, va="bottom", ha="left",
    fontweight="bold", clip_on=False
)

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=15)
ax.set_ylabel("FP/img  (conf ≥ 0.3)", fontsize=16, color=GRAY)
ax.set_ylim(0, 26)
ax.set_xlim(-0.6, len(scenarios) - 0.25)

ax.tick_params(colors=GRAY, labelsize=15)

legend = ax.legend(fontsize=12, frameon=True, loc="upper left")
legend.get_frame().set_facecolor("none")
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_scene_transfer_fp_own_vs_transferred_wide_transparent.{ext}"
    plt.savefig(
        p,
        bbox_inches="tight",
        transparent=True,
        facecolor="none",
        edgecolor="none"
    )
    print(f"Saved → {p}")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

scenarios   = [f"B{i:02d}" for i in range(1, 10)]
map_clean   = [0.895, 0.791, 0.920, 0.903, 0.781, 0.731, 0.671, 0.763, 0.803]
map_patched = [0.801, 0.427, 0.516, 0.244, 0.636, 0.545, 0.547, 0.309, 0.401]

mean_c = np.mean(map_clean)
mean_p = np.mean(map_patched)

x = np.arange(len(scenarios))
w = 0.36

fig, ax = plt.subplots(figsize=(12, 5.2))

bars_c = ax.bar(x - w/2, map_clean,   width=w, color=LIGHT_BLUE,
                edgecolor=BLUE, linewidth=1.6, label="Clean",   zorder=3)
bars_p = ax.bar(x + w/2, map_patched, width=w, color=RED,
                edgecolor=DARK_RED, linewidth=1.6, label="Patched", zorder=3)

# Value labels above patched bars only
for bar, v in zip(bars_p, map_patched):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.013,
            f"{v:.3f}", ha="center", va="bottom",
            fontsize=8, color=DARK_RED, fontweight="bold")

# Mean dashed lines
ax.axhline(mean_c, color=BLUE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_p, color=RED,  lw=1.5, ls="--", alpha=0.65, zorder=2)

# Mean labels — outside right edge, both above their lines
ax.text(8.45, mean_c + 0.014, f"mean = {mean_c:.3f}",
        fontsize=13, color=BLUE, va="bottom", ha="left", fontweight="bold",
        clip_on=False)
ax.text(8.45, mean_p + 0.014, f"mean = {mean_p:.3f}",
        fontsize=13, color=RED, va="bottom", ha="left", fontweight="bold",
        clip_on=False)

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=15)
ax.set_ylabel("mAP50", fontsize=16, color=GRAY)
ax.set_ylim(0, 1.05)

ax.tick_params(colors=GRAY, labelsize=15)
ax.legend(fontsize=14, frameon=False, loc="upper right",
          bbox_to_anchor=(1.0, 0.98))
ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()
for ext in ("png", "pdf"):
    p = f"/content/figures/fig_map_clean_patched.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-scenario transferability — false positives only
# Patch1 optimised on billboard01, evaluated on billboard02–09
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

scenarios = [f"B{i:02d}" for i in range(2, 10)]

fp_clean = np.array([0.40, 0.18, 0.35, 1.60, 0.82, 0.62, 0.70, 2.20])
fp_patch = np.array([9.20, 7.40, 11.80, 7.60, 6.80, 12.60, 14.40, 8.20])

mean_c = np.mean(fp_clean)
mean_p = np.mean(fp_patch)

x = np.arange(len(scenarios))
w = 0.34

fig, ax = plt.subplots(figsize=(6.2, 5.2))

ax.bar(x - w/2, fp_clean, width=w,
       color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
       label="Clean", zorder=3)

ax.bar(x + w/2, fp_patch, width=w,
       color=RED, edgecolor=DARK_RED, linewidth=1.6,
       label="Patch1 transferred", zorder=3)

# Mean dashed lines — same style as your mAP plot
ax.axhline(mean_c, color=BLUE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_p, color=RED,  lw=1.5, ls="--", alpha=0.65, zorder=2)

# Mean labels — same style as your mAP plot
ax.text(7.45, mean_c + 0.30, f"mean = {mean_c:.2f}",
        fontsize=13, color=BLUE, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.text(7.45, mean_p + 0.30, f"mean = {mean_p:.2f}",
        fontsize=13, color=RED, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=15)

# Y-axis kept like before
ax.set_ylabel("FP/img  (τ = 0.5)", fontsize=16, color=GRAY)

ax.set_ylim(0, 16)
ax.tick_params(colors=GRAY, labelsize=15)

legend = ax.legend(fontsize=12, frameon=True, loc="upper left")
legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_scene_transfer_patch1_fp_only.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-scenario transferability — FP/image only
# Clean vs own scenario-specific patch vs transferred Patch1
# Source: thesis Figure 6.6, conf ≥ 0.3
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"
ORANGE     = "#F28E2B"
DARK_ORANGE = "#A95500"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

# B02–B09 only: Patch1 was trained on B01 and transferred to these scenes
scenarios = [f"B{i:02d}" for i in range(2, 10)]

# Figure 6.6 values: FP/image @ conf ≥ 0.3
fp_clean = np.array([0.6, 0.2, 0.4, 0.4, 0.5, 1.2, 0.1, 1.0])
fp_own   = np.array([17.3, 18.6, 19.3, 24.2, 22.6, 17.9, 15.2, 22.0])  # scenario-specific / own patch
fp_trans = np.array([11.9, 14.7, 14.7, 18.5, 18.1, 15.7, 13.9, 19.7])  # Patch1 transferred from B01

mean_c = np.mean(fp_clean)
mean_o = np.mean(fp_own)
mean_t = np.mean(fp_trans)

x = np.arange(len(scenarios))
w = 0.25

fig, ax = plt.subplots(figsize=(6.8, 5.2))

# Bars
ax.bar(x - w, fp_clean, width=w,
       color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
       label="Clean", zorder=3)

ax.bar(x, fp_own, width=w,
       color=ORANGE, edgecolor=DARK_ORANGE, linewidth=1.6,
       label="Own patch", zorder=3)

ax.bar(x + w, fp_trans, width=w,
       color=RED, edgecolor=DARK_RED, linewidth=1.6,
       label="Patch1 transferred", zorder=3)

# Mean dashed lines — same style as your mAP plot
ax.axhline(mean_c, color=BLUE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_o, color=ORANGE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_t, color=RED, lw=1.5, ls="--", alpha=0.65, zorder=2)

# Mean labels — outside right edge, above lines
ax.text(7.45, mean_c + 0.45, f"mean = {mean_c:.1f}",
        fontsize=13, color=BLUE, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.text(7.45, mean_o + 0.45, f"mean = {mean_o:.1f}",
        fontsize=13, color=ORANGE, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.text(7.45, mean_t + 0.45, f"mean = {mean_t:.1f}",
        fontsize=13, color=RED, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

# Axes
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=15)
ax.set_ylabel("FP/img  (conf ≥ 0.3)", fontsize=16, color=GRAY)

# Own patch values go above 16, so use 0–26 for this three-bar comparison
ax.set_ylim(0, 26)

ax.tick_params(colors=GRAY, labelsize=15)

# Legend box
legend = ax.legend(fontsize=12, frameon=True, loc="upper left")
legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

# Styling
ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_scene_transfer_fp_own_vs_transferred.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-scenario transferability — FP/image only
# Clean vs own scenario-specific patch vs transferred Patch1
# Source: thesis Figure 6.6, conf ≥ 0.3
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"
ORANGE     = "#F28E2B"
DARK_ORANGE = "#A95500"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

scenarios = [f"B{i:02d}" for i in range(2, 10)]

fp_clean = np.array([0.6, 0.2, 0.4, 0.4, 0.5, 1.2, 0.1, 1.0])
fp_own   = np.array([17.3, 18.6, 19.3, 24.2, 22.6, 17.9, 15.2, 22.0])
fp_trans = np.array([11.9, 14.7, 14.7, 18.5, 18.1, 15.7, 13.9, 19.7])

mean_c = np.mean(fp_clean)
mean_o = np.mean(fp_own)
mean_t = np.mean(fp_trans)

x = np.arange(len(scenarios))

# Wider grouped bars
w = 0.29

# Wider figure for PowerPoint
fig, ax = plt.subplots(figsize=(9.2, 5.2))

ax.bar(x - w, fp_clean, width=w,
       color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
       label="Clean", zorder=3)

ax.bar(x, fp_own, width=w,
       color=ORANGE, edgecolor=DARK_ORANGE, linewidth=1.6,
       label="Own patch", zorder=3)

ax.bar(x + w, fp_trans, width=w,
       color=RED, edgecolor=DARK_RED, linewidth=1.6,
       label="Patch1 transferred", zorder=3)

# Mean dashed lines
ax.axhline(mean_c, color=BLUE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_o, color=ORANGE, lw=1.5, ls="--", alpha=0.65, zorder=2)
ax.axhline(mean_t, color=RED, lw=1.5, ls="--", alpha=0.65, zorder=2)

# Mean labels — outside right edge
ax.text(7.55, mean_c + 0.45, f"mean = {mean_c:.1f}",
        fontsize=13, color=BLUE, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.text(7.55, mean_o + 0.45, f"mean = {mean_o:.1f}",
        fontsize=13, color=ORANGE, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.text(7.55, mean_t + 0.45, f"mean = {mean_t:.1f}",
        fontsize=13, color=RED, va="bottom", ha="left",
        fontweight="bold", clip_on=False)

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=15)
ax.set_ylabel("FP/img  (conf ≥ 0.3)", fontsize=16, color=GRAY)
ax.set_ylim(0, 26)

# Add a bit of right-side room for mean labels
ax.set_xlim(-0.6, len(scenarios) - 0.25)

ax.tick_params(colors=GRAY, labelsize=15)

legend = ax.legend(fontsize=12, frameon=True, loc="upper left")
legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

ax.grid(axis="y", color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_scene_transfer_fp_own_vs_transferred_wide.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-architecture transferability — clean slide version
# Source: Table 6.8 — mean across 9 billboard scenarios
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

models = [
    "YOLOv8n\nwhite-box",
    "YOLOv8s",
    "YOLOv5n",
    "RT-DETR-L",
    "NanoDet-Plus"
]

map_clean = np.array([0.806, 0.890, 0.861, 0.904, 0.646])
map_patch = np.array([0.492, 0.632, 0.567, 0.678, 0.584])

fp_clean  = np.array([0.55, 0.50, 0.56, 0.86, 1.29])
fp_patch  = np.array([20.08, 0.88, 2.42, 2.79, 1.82])

delta_map = map_patch - map_clean

y = np.arange(len(models))
h = 0.34

fig, (ax_map, ax_fp) = plt.subplots(1, 2, figsize=(13.2, 5.4))
fig.patch.set_facecolor(WHITE)

# Left plot: mAP50
ax_map.barh(
    y + h/2, map_clean, height=h,
    color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.5,
    label="Clean", zorder=3
)

ax_map.barh(
    y - h/2, map_patch, height=h,
    color=RED, edgecolor=DARK_RED, linewidth=1.5,
    label="Patched", zorder=3
)

ax_map.set_yticks(y)
ax_map.set_yticklabels(models, fontsize=14)
ax_map.set_xlabel("mAP50", fontsize=15, color=GRAY)
ax_map.set_xlim(0, 1.05)
ax_map.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))

ax_map.tick_params(colors=GRAY, labelsize=13)
ax_map.grid(axis="x", color="#EEEEEE", lw=1.0, zorder=0)
ax_map.spines["left"].set_color("#DDDDDD")
ax_map.spines["bottom"].set_color("#DDDDDD")

for i, d in enumerate(delta_map):
    ax_map.text(
        1.02, y[i],
        f"Δ {d:.3f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color=DARK_RED if abs(d) > 0.20 else MID_GRAY,
        clip_on=False
    )

# Right plot: false positives
ax_fp.barh(
    y + h/2, fp_clean, height=h,
    color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.5,
    label="Clean", zorder=3
)

ax_fp.barh(
    y - h/2, fp_patch, height=h,
    color=RED, edgecolor=DARK_RED, linewidth=1.5,
    label="Patched", zorder=3
)

ax_fp.set_yticks(y)
ax_fp.set_yticklabels(models, fontsize=14)
ax_fp.set_xlabel("FP/img  (conf ≥ 0.3)", fontsize=15, color=GRAY)
ax_fp.set_xlim(0, 21)

ax_fp.tick_params(colors=GRAY, labelsize=13)
ax_fp.grid(axis="x", color="#EEEEEE", lw=1.0, zorder=0)
ax_fp.spines["left"].set_color("#DDDDDD")
ax_fp.spines["bottom"].set_color("#DDDDDD")

for i, v in enumerate(fp_patch):
    ax_fp.text(
        v + 0.35, y[i] - h/2,
        f"{v:.2f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color=DARK_RED,
        clip_on=False
    )

# Shared legend — centered at bottom
handles, labels = ax_map.get_legend_handles_labels()

legend = fig.legend(
    handles,
    labels,
    fontsize=13,
    frameon=True,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2
)

legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

plt.tight_layout(rect=[0, 0.10, 1, 1], w_pad=3.5)

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_arch_transfer_clean_no_titles.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-architecture transferability — clean 2-plot slide version
# Source: Table 6.8 — mean across 9 billboard scenarios
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

# Order: white-box first, then transfer models
models = [
    "YOLOv8n\nwhite-box",
    "YOLOv8s",
    "YOLOv5n",
    "RT-DETR-L",
    "NanoDet-\nPlus"
]

map_clean = np.array([0.806, 0.890, 0.861, 0.904, 0.646])
map_patch = np.array([0.492, 0.632, 0.567, 0.678, 0.584])

fp_clean  = np.array([0.55, 0.50, 0.56, 0.86, 1.29])
fp_patch  = np.array([20.08, 0.88, 2.42, 2.79, 1.82])

delta_map = map_patch - map_clean

y = np.arange(len(models))
h = 0.34

fig, (ax_map, ax_fp) = plt.subplots(1, 2, figsize=(13.4, 5.6))
fig.patch.set_facecolor(WHITE)

# ── Left plot: mAP50 ────────────────────────────────────────────────────────
ax_map.barh(
    y + h/2, map_clean, height=h,
    color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
    label="Clean", zorder=3
)

ax_map.barh(
    y - h/2, map_patch, height=h,
    color=RED, edgecolor=DARK_RED, linewidth=1.6,
    label="Patched", zorder=3
)

ax_map.set_yticks(y)
ax_map.set_yticklabels(models, fontsize=15)
ax_map.set_xlabel("mAP50", fontsize=16, color=GRAY)
ax_map.set_xlim(0, 1.05)
ax_map.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
ax_map.invert_yaxis()

ax_map.tick_params(colors=GRAY, labelsize=14)
ax_map.grid(axis="x", color="#EEEEEE", lw=1.0, zorder=0)
ax_map.spines["left"].set_color("#DDDDDD")
ax_map.spines["bottom"].set_color("#DDDDDD")

# Δ annotations
for i, d in enumerate(delta_map):
    ax_map.text(
        1.015, y[i],
        f"Δ {d:.3f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color=DARK_RED if abs(d) > 0.20 else MID_GRAY,
        clip_on=False
    )

# ── Right plot: FP/img ──────────────────────────────────────────────────────
ax_fp.barh(
    y + h/2, fp_clean, height=h,
    color=LIGHT_BLUE, edgecolor=BLUE, linewidth=1.6,
    label="Clean", zorder=3
)

ax_fp.barh(
    y - h/2, fp_patch, height=h,
    color=RED, edgecolor=DARK_RED, linewidth=1.6,
    label="Patched", zorder=3
)

ax_fp.set_yticks(y)
ax_fp.set_yticklabels(models, fontsize=15)
ax_fp.set_xlabel("FP/img  (conf ≥ 0.3)", fontsize=16, color=GRAY)
ax_fp.set_xlim(0, 21.5)
ax_fp.invert_yaxis()

ax_fp.tick_params(colors=GRAY, labelsize=14)
ax_fp.grid(axis="x", color="#EEEEEE", lw=1.0, zorder=0)
ax_fp.spines["left"].set_color("#DDDDDD")
ax_fp.spines["bottom"].set_color("#DDDDDD")

# Value labels for patched FP
for i, v in enumerate(fp_patch):
    ax_fp.text(
        v + 0.35, y[i] - h/2,
        f"{v:.2f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color=DARK_RED,
        clip_on=False
    )

# Shared legend centered at bottom
handles, labels = ax_map.get_legend_handles_labels()
legend = fig.legend(
    handles, labels,
    fontsize=13,
    frameon=True,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2
)
legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

plt.tight_layout(rect=[0, 0.09, 1, 1], w_pad=3.6)

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_cross_arch_transfer_2plots_clean.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ============================================================
# PREVIEW Figures 6.9 and 6.11 before saving/exporting
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09"]

latency_margin = np.array([
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],
])

power_margin = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.15, 0.51, 0.00],
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.19, 0.04, 0.01],
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.02, 0.00, 0.12],
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.00, 0.22, 0.13],
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.35, 0.21, 0.11],
])

def preview_vector_heatmap(
    data,
    title,
    colorbar_label,
    vmax,
    threshold=1.0,
):
    fig, ax = plt.subplots(figsize=(7.2, 2.8), constrained_layout=True)

    cmap = plt.get_cmap("Blues")
    norm = Normalize(vmin=0.0, vmax=vmax)

    n_rows, n_cols = data.shape

    for i in range(n_rows):
        for j in range(n_cols):
            value = float(data[i, j])

            rect = Rectangle(
                (j, i),
                1,
                1,
                facecolor=cmap(norm(value)),
                edgecolor="white",
                linewidth=0.8,
            )
            ax.add_patch(rect)

            if value >= threshold:
                ax.add_patch(
                    Rectangle(
                        (j, i),
                        1,
                        1,
                        facecolor="none",
                        edgecolor="black",
                        linewidth=1.2,
                        hatch="///",
                    )
                )

            text_color = "white" if norm(value) > 0.58 else "black"

            ax.text(
                j + 0.5,
                i + 0.5,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=text_color,
            )

    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)

    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels(scenarios)

    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(models)

    ax.set_xlabel("Billboard scenario")
    ax.set_title(title, pad=8)

    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(colorbar_label)

    plt.show()

preview_vector_heatmap(
    latency_margin,
    title="Figure 6.9: Latency visibility margin",
    colorbar_label="Margin ratio",
    vmax=1.25,
)

preview_vector_heatmap(
    power_margin,
    title="Figure 6.11: Power visibility margin",
    colorbar_label="Margin ratio",
    vmax=2.0,
)

In [ ]:
# ============================================================
# Create vector SVG/PDF heatmaps for Thesis Figures 6.9 and 6.11
# Google Colab single-cell version
# ============================================================

import os
import zipfile
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize

# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
OUT_SVG = Path("figures_svg")
OUT_PDF = Path("figures_pdf")
OUT_PNG = Path("figures_png_backup")

OUT_SVG.mkdir(exist_ok=True)
OUT_PDF.mkdir(exist_ok=True)
OUT_PNG.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Matplotlib vector export settings
# ------------------------------------------------------------
mpl.rcParams.update({
    "svg.fonttype": "none",   # editable text in SVG
    "pdf.fonttype": 42,       # proper embedded fonts in PDF
    "ps.fonttype": 42,
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09"]

# ------------------------------------------------------------
# Figure 6.9 data: latency visibility margin
# |ΔE2E| / σ_clean
# ------------------------------------------------------------
latency_margin = np.array([
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],
])

# ------------------------------------------------------------
# Figure 6.11 data: power visibility margin
# |ΔP| / 0.22 W
# ------------------------------------------------------------
power_margin = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.15, 0.51, 0.00],
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.19, 0.04, 0.01],
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.02, 0.00, 0.12],
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.00, 0.22, 0.13],
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.35, 0.21, 0.11],
])

def draw_vector_heatmap(
    data,
    filename,
    colorbar_label,
    vmax,
    threshold=1.0,
):
    """
    Draw heatmap using vector rectangles, not imshow().
    This keeps the SVG/PDF truly vector.
    """

    fig, ax = plt.subplots(figsize=(6.4, 2.45), constrained_layout=True)

    cmap = plt.get_cmap("Blues")
    norm = Normalize(vmin=0.0, vmax=vmax)

    n_rows, n_cols = data.shape

    for i in range(n_rows):
        for j in range(n_cols):
            value = float(data[i, j])

            rect = Rectangle(
                (j, i),
                1,
                1,
                facecolor=cmap(norm(value)),
                edgecolor="white",
                linewidth=0.8,
            )
            ax.add_patch(rect)

            # Hatch values above the visibility threshold
            if value >= threshold:
                ax.add_patch(
                    Rectangle(
                        (j, i),
                        1,
                        1,
                        facecolor="none",
                        edgecolor="black",
                        linewidth=1.2,
                        hatch="///",
                    )
                )

            text_color = "white" if norm(value) > 0.58 else "black"

            ax.text(
                j + 0.5,
                i + 0.5,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color=text_color,
            )

    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)

    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels(scenarios)

    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(models)

    ax.set_xlabel("Billboard scenario")
    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(colorbar_label)

    svg_path = OUT_SVG / f"{filename}.svg"
    pdf_path = OUT_PDF / f"{filename}.pdf"
    png_path = OUT_PNG / f"{filename}.png"

    fig.savefig(svg_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved SVG: {svg_path}")
    print(f"Saved PDF: {pdf_path}")
    print(f"Saved PNG: {png_path}")
    print()

# ------------------------------------------------------------
# Generate Figure 6.9
# ------------------------------------------------------------
draw_vector_heatmap(
    data=latency_margin,
    filename="figure_6_9_latency_visibility_margin",
    colorbar_label="Margin ratio",
    vmax=1.25,
    threshold=1.0,
)

# ------------------------------------------------------------
# Generate Figure 6.11
# ------------------------------------------------------------
draw_vector_heatmap(
    data=power_margin,
    filename="figure_6_11_power_visibility_margin",
    colorbar_label="Margin ratio",
    vmax=2.0,
    threshold=1.0,
)

# ------------------------------------------------------------
# Check whether SVGs contain raster <image> tags
# Good result: "OK: no raster image tags found"
# ------------------------------------------------------------
for svg_file in OUT_SVG.glob("*.svg"):
    text = svg_file.read_text(encoding="utf-8")
    if "<image" in text:
        print(f"WARNING: {svg_file} contains raster image tags.")
    else:
        print(f"OK: {svg_file} has no raster image tags.")

# ------------------------------------------------------------
# Zip everything for download
# ------------------------------------------------------------
zip_name = "thesis_fixed_vector_figures.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [OUT_SVG, OUT_PDF, OUT_PNG]:
        for file in folder.glob("*"):
            z.write(file, arcname=f"{folder.name}/{file.name}")

print()
print(f"Created ZIP: {zip_name}")

# ------------------------------------------------------------
# Download automatically in Colab
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Not running in Colab, so automatic download was skipped.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cleaner dual heatmap for hardware invisibility
# No titles, no bottom notes
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.dpi": 150,
    "savefig.dpi": 220,
    "savefig.facecolor": WHITE,
})

models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet-\nPlus", "RT-DETR-L"]
scenarios = [f"B0{i}" for i in range(1, 10)]

# Values from thesis
latency = np.array([
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],
])

power = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.12, 0.51, 0.00],
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.10, 0.04, 0.01],
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.01, 0.00, 0.12],
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.01, 0.22, 0.13],
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.10, 0.21, 0.11],
])

# Soft colormap: light blue -> off-white -> soft red
cmap = LinearSegmentedColormap.from_list(
    "soft_visibility",
    ["#BFD4EA", "#F5F5F5", "#E34A4A"]
)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
fig.patch.set_facecolor(WHITE)

data_list = [latency, power]

for ax, data in zip(axes, data_list):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=2.0, aspect="auto")

    # Ticks
    ax.set_xticks(np.arange(len(scenarios)))
    ax.set_xticklabels(scenarios, fontsize=13, color=GRAY)
    ax.set_yticks(np.arange(len(models)))
    ax.set_yticklabels(models, fontsize=14, color=GRAY)

    # Grid lines between cells
    ax.set_xticks(np.arange(-.5, len(scenarios), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(models), 1), minor=True)
    ax.grid(which="minor", color=WHITE, linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Remove spines
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Cell text
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            is_visible = v >= 1.0
            ax.text(
                j, i, f"{v:.2f}",
                ha="center", va="center",
                fontsize=12,
                fontweight="bold" if is_visible else "normal",
                color=DARK_RED if is_visible else GRAY
            )

# Highlight only the visible case in the power plot: RT-DETR-L, B03
# row index 4, col index 2
rect = patches.Rectangle(
    (2 - 0.5, 4 - 0.5), 1, 1,
    linewidth=3.0, edgecolor=RED, facecolor="none"
)
axes[1].add_patch(rect)

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_hardware_invisibility_clean_no_titles.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cleaner dual heatmap for hardware invisibility
# With simplified legend
# Right heatmap has no repeated model labels
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import os

os.makedirs("/content/figures", exist_ok=True)

RED        = "#CC0000"
DARK_RED   = "#880000"
BLUE       = "#1A4FA0"
LIGHT_BLUE = "#C8DDF5"
GRAY       = "#333333"
MID_GRAY   = "#777777"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.dpi": 150,
    "savefig.dpi": 220,
    "savefig.facecolor": WHITE,
})

models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet-\nPlus", "RT-DETR-L"]
scenarios = [f"B0{i}" for i in range(1, 10)]

# Values from thesis
latency = np.array([
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],
])

power = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.12, 0.51, 0.00],
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.10, 0.04, 0.01],
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.01, 0.00, 0.12],
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.01, 0.22, 0.13],
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.10, 0.21, 0.11],
])

# Soft colormap: light blue → off-white → soft red
cmap = LinearSegmentedColormap.from_list(
    "soft_visibility",
    ["#BFD4EA", "#F5F5F5", "#E34A4A"]
)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.4))
fig.patch.set_facecolor(WHITE)

data_list = [latency, power]

for idx, (ax, data) in enumerate(zip(axes, data_list)):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=2.0, aspect="auto")

    # X ticks
    ax.set_xticks(np.arange(len(scenarios)))
    ax.set_xticklabels(scenarios, fontsize=13, color=GRAY)

    # Y ticks: show model names only on the left heatmap
    ax.set_yticks(np.arange(len(models)))

    if idx == 0:
        ax.set_yticklabels(models, fontsize=14, color=GRAY)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)

    # Grid lines between cells
    ax.set_xticks(np.arange(-.5, len(scenarios), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(models), 1), minor=True)
    ax.grid(which="minor", color=WHITE, linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Remove spines
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Cell values
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            is_visible = v >= 1.0
            ax.text(
                j, i, f"{v:.2f}",
                ha="center", va="center",
                fontsize=12,
                fontweight="bold" if is_visible else "normal",
                color=DARK_RED if is_visible else GRAY
            )

# Highlight only the visible case in the power plot: RT-DETR-L, B03
rect = patches.Rectangle(
    (2 - 0.5, 4 - 0.5), 1, 1,
    linewidth=3.0,
    edgecolor=RED,
    facecolor="none"
)
axes[1].add_patch(rect)

# Simplified legend
legend_handles = [
    patches.Patch(facecolor="#BFD4EA", edgecolor="none", label="Below threshold (< 1)"),
    patches.Patch(facecolor="#E34A4A", edgecolor="none", label="Above threshold (≥ 1)"),
]

legend = fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2,
    frameon=True,
    fontsize=12
)

legend.get_frame().set_facecolor(WHITE)
legend.get_frame().set_edgecolor("#DDDDDD")
legend.get_frame().set_linewidth(1.0)

plt.tight_layout(rect=[0, 0.10, 1, 1])

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_hardware_invisibility_clean_with_simple_legend.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = "/content/figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Palette ───────────────────────────────────────────────────────────────────
C = dict(
    red       = "#CC0000",
    dark_red  = "#880000",
    blue      = "#1A4FA0",
    light_blue= "#C8DDF5",
    gray      = "#333333",
    spine     = "#BBBBBB",
    grid      = "#E5E5E5",
    guide     = "#888888",
    white     = "#FFFFFF",
)

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"        : "DejaVu Sans",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.facecolor"  : "none",
})

# ── Data ──────────────────────────────────────────────────────────────────────
np.random.seed(7)
x_in,  y_in  = np.random.uniform(-0.72, 0.72, (2, 44))
x_out, y_out = np.array([1.14]), np.array([1.12])

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4.7, 4.7))
fig.patch.set_alpha(0)
ax.set_facecolor("none")

# Invisible region box
ax.add_patch(patches.Rectangle(
    (-1, -1), 2, 2,
    linewidth=2.3, edgecolor=C["blue"],
    facecolor=C["light_blue"], alpha=0.32, zorder=1,
))

# Scatter points
scatter_kw = dict(edgecolors=C["white"], linewidths=0.8)
ax.scatter(x_in,  y_in,  s=48,  color=C["blue"], alpha=0.96, label="Invisible", zorder=4, **scatter_kw)
ax.scatter(x_out, y_out, s=110, color=C["red"],              label="Visible",   zorder=5, **scatter_kw)

# Crosshair guides
for fn in (ax.axhline, ax.axvline):
    fn(0, color=C["guide"], lw=1.2, linestyle="--", zorder=2)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_xlim(-1.08, 1.62)
ax.set_ylim(-1.08, 1.35)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])

label_kw = dict(fontsize=16, fontweight="bold", color=C["blue"])
ax.set_xlabel("Latency Δ", labelpad=2,  **label_kw)
ax.set_ylabel("Power Δ",   labelpad=8,  **label_kw)

ax.tick_params(axis="both", labelsize=13, colors=C["gray"], pad=2)
ax.grid(color=C["grid"], lw=0.9, zorder=0)

for spine in ("left", "bottom"):
    ax.spines[spine].set(color=C["spine"], linewidth=1.2)

# ── Annotations ───────────────────────────────────────────────────────────────
ax.text(0, 1.08, "Invisible region",
        ha="center", va="bottom",
        fontsize=17, fontweight="bold", color=C["blue"])

ax.annotate("Visible",
    xy=(x_out[0], y_out[0]), xytext=(x_out[0] + 0.22, y_out[0] + 0.18),
    fontsize=15, fontweight="bold", color=C["dark_red"], ha="left", va="bottom",
    arrowprops=dict(
        arrowstyle="-|>",
        lw=2.0,
        color=C["dark_red"],
        connectionstyle="arc3,rad=-0.15",
        shrinkA=3, shrinkB=6,
    ),
)



# ── Save ──────────────────────────────────────────────────────────────────────
plt.subplots_adjust(left=0.12, right=0.78, bottom=0.06, top=0.96)

for ext in ("png", "pdf"):
    path = f"{OUT_DIR}/fig_hardware_invisibility_poster_compact.{ext}"
    plt.savefig(path, bbox_inches="tight", transparent=True,
                facecolor="none", edgecolor="none", pad_inches=0)
    print(f"Saved → {path}")

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Defence trade-off plot — clean version with arrow kept
# Held-out billboard04–billboard09
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figures", exist_ok=True)

RED         = "#CC0000"
DARK_RED    = "#880000"
ORANGE      = "#FF8800"
GRAY        = "#333333"
MID_GRAY    = "#999999"
LIGHT_GRAY  = "#BFBFBF"
WHITE       = "#FFFFFF"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       220,
    "savefig.facecolor": WHITE,
})

# Data
defences = ["None", "JPEG", "Bit-depth", "Gaussian", "Median", "SODA", "LGS"]

patched_map50 = np.array([
    0.254,  # None
    0.251,  # JPEG
    0.254,  # Bit-depth
    0.135,  # Gaussian
    0.163,  # Median
    0.231,  # SODA
    0.235,  # LGS
])

fp_suppression = np.array([
    0.000,    # None
    0.633,    # JPEG
   -0.067,    # Bit-depth
    23.560,   # Gaussian
    29.227,   # Median
    32.690,   # SODA
    40.540,   # LGS
])

styles = {
    "None":      {"color": MID_GRAY, "marker": "s", "size": 95},
    "JPEG":      {"color": MID_GRAY, "marker": "s", "size": 95},
    "Bit-depth": {"color": MID_GRAY, "marker": "s", "size": 95},
    "Gaussian":  {"color": ORANGE,   "marker": "^", "size": 145},
    "Median":    {"color": ORANGE,   "marker": "^", "size": 145},
    "SODA":      {"color": DARK_RED, "marker": "o", "size": 190},
    "LGS":       {"color": RED,      "marker": "o", "size": 220},
}

fig, ax = plt.subplots(figsize=(10.2, 5.8))
fig.patch.set_facecolor(WHITE)

# Plot points
for d, x, y in zip(defences, patched_map50, fp_suppression):
    st = styles[d]
    ax.scatter(
        x, y,
        s=st["size"],
        color=st["color"],
        marker=st["marker"],
        edgecolors=WHITE if d in ["LGS", "SODA"] else st["color"],
        linewidth=1.8,
        zorder=4
    )

# Labels
offsets = {
    "None":      (0.004, -1.9),
    "JPEG":      (0.002,  1.0),
    "Bit-depth": (-0.025, -2.0),
    "Gaussian":  (-0.018, 1.2),
    "Median":    (-0.011, 1.4),
    "SODA":      (0.004, -3.8),
    "LGS":       (0.004, 1.5),
}

for d, x, y in zip(defences, patched_map50, fp_suppression):
    ox, oy = offsets[d]
    st = styles[d]
    ax.text(
        x + ox,
        y + oy,
        d,
        fontsize=15 if d in ["LGS", "SODA"] else 13,
        color=st["color"],
        fontweight="bold" if d in ["LGS", "SODA"] else "normal",
        va="bottom"
    )

# Keep arrow, remove "better" text
ax.annotate(
    "",
    xy=(0.265, 43.5),
    xytext=(0.125, 10.0),
    arrowprops=dict(
        arrowstyle="-|>",
        color=LIGHT_GRAY,
        lw=1.8,
        shrinkA=0,
        shrinkB=0
    ),
    zorder=1
)

# Axes
ax.set_xlabel("Patched mAP50  (accuracy preserved →)", fontsize=16, color=GRAY)
ax.set_ylabel("FP/img suppression  (overload suppressed ↑)", fontsize=16, color=GRAY)

ax.set_xlim(0.115, 0.270)
ax.set_ylim(-3, 45)

ax.tick_params(colors=GRAY, labelsize=14)
ax.grid(color="#EEEEEE", lw=1.0, zorder=0)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()

for ext in ("png", "pdf"):
    p = f"/content/figures/fig_defence_tradeoff_clean_with_arrow.{ext}"
    plt.savefig(p, bbox_inches="tight", facecolor=WHITE)
    print(f"Saved → {p}")

plt.show()